# LightGBM Pipeline V1
Production-ready starter notebook based on your Logistic Regression pipeline.

In [13]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, balanced_accuracy_score, precision_score, recall_score, f1_score
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from joblib import dump

In [25]:
class DateFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self,date_column="Date"):
        self.date_column=date_column
        
    def fit(self,X,y=None):
        d=pd.to_datetime(X[self.date_column],errors="coerce")
        self.median_date_=d.dropna().median()
        return self
    
    def transform(self,X):
        X=X.copy()
        X[self.date_column]=pd.to_datetime(X[self.date_column],errors="coerce")
        X["DateMissing"]=X[self.date_column].isna().astype(int)
        X[self.date_column]=X[self.date_column].fillna(self.median_date_)
        # Create categorical features as pandas categorical dtype
        X["Date_Month"] = X[self.date_column].dt.month.astype('category')
        X["Date_DayOfWeek"] = X[self.date_column].dt.dayofweek.astype('category')
        X["Date_DayOfMonth"] = X[self.date_column].dt.day.astype('category')
        X.drop(columns=[self.date_column],inplace=True)
        return X

class NumericFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns if columns else ["Value", "No_Wall_Types"]
        
    def fit(self, X, y=None):
        self.medians_ = {}
        for col in self.columns:
            if col in X.columns:
                self.medians_[col] = X[col].median()
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # Impute all numeric columns
        for col in self.columns:
            if col in X.columns:
                X[col] = X[col].fillna(self.medians_[col])
        
        # Log transform only Value (skewed)
        if "Value" in X.columns:
            X["Value"] = np.log1p(X["Value"])
            
        return X

class EstimatorFeatureTransformer(BaseEstimator,TransformerMixin):
    def __init__(self,column="Priced_By"):
        self.column=column

    def fit(self,X,y=None): 
        return self
    
    def transform(self,X):
        X=X.copy()
        s=X[self.column].fillna("MISSING").astype(str)
        X["EstimatorCount"]=s.str.split("/").apply(len)
        X.loc[s.str.upper().eq("MISSING"),"EstimatorCount"]=0
        X["EstimatorCount"]=X["EstimatorCount"].astype(int)
        X["EstimatorMissing"]=s.str.upper().eq("MISSING").astype(int)
        return X

# Safe categorical filler that converts to pandas categorical dtype
class CategoricalImputer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            if col in X.columns:
                # Fill missing and convert to categorical
                X[col] = X[col].fillna("MISSING").astype('category')
        return X 
    

In [ ]:
# Load your dataframe into quotation_data_df before running
quotation_data_path = r"C:\Users\Phong\OneDrive - ICB Construction\Phong\data\Python_ETL\DS\ML_Models\data\Quotation Data.xlsx"
quotation_data_df = pd.read_excel(quotation_data_path)

raw_features=[
"Date","Value","No_Wall_Types","Priced_By","Suburb","Client_Clean",
"Timber_RW","RC_Pile","Steel_Beam","Sheetpile","Anchor","Block",
"Shotcrete","Capping_Beam","Earthwork","Concrete_Slab","Precast",
"Culvert","Slip_Repair","Soil_Nail","Rock_RW","Bridge","Concrete",
"Design_and_Build","Budget","Drill_Only","Labour_Only",
"Driven_Pile","Palisade","Boardwalk","Soldier","Insitu",
"Barrier","Noise_RW","Base","Casing","Crib",
"DayWork","Flood_Repair","Micro_Pile","Reno","Temp_RW","Other"]

target="Success"

quotation_data_df["Date"]=pd.to_datetime(quotation_data_df["Date"],errors="coerce")
quotation_data_df=quotation_data_df.sort_values("Date").reset_index(drop=True)

X=quotation_data_df[raw_features].copy()
y=quotation_data_df[target]

split=int(len(X)*0.8)
X_train,X_test=X.iloc[:split],X.iloc[split:]
y_train,y_test=y.iloc[:split],y.iloc[split:]

# Define which columns will be categorical after transformation

model = LGBMClassifier(
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

# Pipeline encapsulates engineering steps to completely stop data leakage
lightgbm_pipeline = Pipeline([
    ("date", DateFeatureTransformer()),
    ("estimator", EstimatorFeatureTransformer(column="Priced_By")),
    ("numeric", NumericFeatureTransformer(columns=["Value", "No_Wall_Types"])),
    ("cat_imputer", CategoricalImputer(columns=["Priced_By","Suburb", "Client_Clean"])),
    ("classifier", model)
])

param_dist = {
    "classifier__num_leaves": [31, 50, 70, 100],
    "classifier__max_depth": [-1, 8, 12, 16],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "classifier__n_estimators": [300, 500, 800, 1000],
    "classifier__min_child_samples": [10, 20, 30, 50],
    "classifier__subsample": [0.6, 0.8, 1.0],
    "classifier__colsample_bytree": [0.6, 0.8, 1.0],
    "classifier__reg_alpha": [0, 0.1, 0.5, 1.0],
    "classifier__reg_lambda": [0, 0.1, 0.5, 1.0],
    "classifier__class_weight": [None, "balanced"]
}

tscv=TimeSeriesSplit(n_splits=5)

search = RandomizedSearchCV(
    lightgbm_pipeline,
    param_dist,
    n_iter=30,
    cv=tscv,
    scoring="balanced_accuracy",
    random_state=42,
    n_jobs=-1,
    verbose=2)

search.fit(X_train,y_train)

best=search.best_estimator_
print("\n" + "="*60)
print("Best Parameters:")
print(search.best_params_)
print(f"Best CV Balanced Accuracy: {search.best_score_:.4f}")
print("="*60)

# Evaluation
# ================================================================
proba=best.predict_proba(X_test)[:,1]
threshold=0.549
pred=(proba>=threshold).astype(int)

print(classification_report(y_test,pred))
print("AUC:",roc_auc_score(y_test,proba))

# Feature Importance
# ================================================================
importance = pd.DataFrame({
    "Feature": best.named_steps['classifier'].feature_name_,
    "Importance": best.named_steps['classifier'].feature_importances_
}).sort_values("Importance", ascending=False)

print("\n" + "="*60)
print("Top 20 Feature Importances:")
print(importance.head(20))
print("="*60)

# Save Model
# ================================================================
dump(best,"LightGBM_V1.joblib")

Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best Parameters:
{'classifier__subsample': 0.8, 'classifier__reg_lambda': 0.5, 'classifier__reg_alpha': 0, 'classifier__num_leaves': 100, 'classifier__n_estimators': 300, 'classifier__min_child_samples': 30, 'classifier__max_depth': 12, 'classifier__learning_rate': 0.05, 'classifier__colsample_bytree': 1.0, 'classifier__class_weight': 'balanced'}
Best CV Balanced Accuracy: 0.6288
              precision    recall  f1-score   support

           0       0.87      0.79      0.82       749
           1       0.34      0.47      0.40       175

    accuracy                           0.73       924
   macro avg       0.60      0.63      0.61       924
weighted avg       0.77      0.73      0.74       924

AUC: 0.6612016021361816

Top 20 Feature Importances:
             Feature  Importance
0              Value       16839
45   Date_DayOfMonth        1499
43        Date_Month        1077
5          Timber_RW        1051
1      No

['LightGBM_V1.joblib']

In [15]:
from sklearn.metrics import precision_recall_curve

# Get probabilities
proba = best.predict_proba(X_test)[:, 1]

# Find optimal threshold using precision-recall curve
precision, recall, thresholds = precision_recall_curve(y_test, proba)

# Calculate F1 for each threshold
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
optimal_idx = np.argmax(f1_scores[:-1])  # -1 to match thresholds length
optimal_threshold = thresholds[optimal_idx]

print(f"Optimal threshold: {optimal_threshold:.3f}")
print(f"F1 at optimal threshold: {f1_scores[optimal_idx]:.3f}")

# Apply custom threshold
pred_custom = (proba >= optimal_threshold).astype(int)
print("\nClassification with custom threshold:")
print(classification_report(y_test, pred_custom))

# Or try multiple thresholds to see the trade-off
thresholds_to_try = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6]
for thresh in thresholds_to_try:
    pred = (proba >= thresh).astype(int)
    print(f"\nThreshold: {thresh}")
    print(f"Class 1 Precision: {precision_score(y_test, pred):.3f}")
    print(f"Class 1 Recall: {recall_score(y_test, pred):.3f}")
    print(f"Class 1 F1: {f1_score(y_test, pred):.3f}")

Optimal threshold: 0.549
F1 at optimal threshold: 0.398

Classification with custom threshold:
              precision    recall  f1-score   support

           0       0.87      0.79      0.82       749
           1       0.34      0.47      0.40       175

    accuracy                           0.73       924
   macro avg       0.60      0.63      0.61       924
weighted avg       0.77      0.73      0.74       924


Threshold: 0.3
Class 1 Precision: 0.249
Class 1 Recall: 0.651
Class 1 F1: 0.360

Threshold: 0.35
Class 1 Precision: 0.248
Class 1 Recall: 0.583
Class 1 F1: 0.348

Threshold: 0.4
Class 1 Precision: 0.269
Class 1 Recall: 0.554
Class 1 F1: 0.363

Threshold: 0.45
Class 1 Precision: 0.293
Class 1 Recall: 0.531
Class 1 F1: 0.378

Threshold: 0.5
Class 1 Precision: 0.311
Class 1 Recall: 0.497
Class 1 F1: 0.382

Threshold: 0.55
Class 1 Precision: 0.338
Class 1 Recall: 0.463
Class 1 F1: 0.390

Threshold: 0.6
Class 1 Precision: 0.340
Class 1 Recall: 0.394
Class 1 F1: 0.365
